In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parent
WENDLER = REPO / "external/llm-latent-language/data/langs"

# Wendler's 2-letter codes → our FLORES-style codes (language_script)
WENDLER_LANGS = {"en": "eng_Latn", "de": "deu_Latn", "fr": "fra_Latn", "ru": "rus_Cyrl", "zh": "cmn_Hans"}

frames = []
for short, code in WENDLER_LANGS.items():
    df = pd.read_csv(WENDLER / short / "clean.csv")
    frames.append(pd.DataFrame({
        "concept_id": df["word_original"].str.strip().str.upper(),   # "cloud" → "CLOUD"
        "lang": code,
        "word": df["word_translation"].str.strip(),
        "source": "wendler2024",
    }))

concepts = pd.concat(frames, ignore_index=True)
concepts[-5:]


In [ ]:
print(concepts.shape)
print(concepts["lang"].value_counts())
print("duplicate (concept, lang)", concepts.duplicated(["concept_id", "lang"]).sum())
concepts[concepts.duplicated(["lang", "word"], keep=False)]
wide = concepts.pivot(index="concept_id", columns="lang", values="word")   
print("concepts present in all 5 languages:", wide.notna().all(axis=1).sum())  
wide.head()

In [ ]:
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B-Base"
tok = AutoTokenizer.from_pretrained(MODEL)

def token_stats(word, tok):
    out = {}
    for variant, text in [("plain", word), ("space", " " + word)]:
        ids = tok.encode(text, add_special_tokens=False)
        first = tok.decode(ids[:1])
        out[f"n_tok_{variant}"] = len(ids)
        out[f"first_id_{variant}"] = ids[0]
        out[f"first_str_{variant}"] = first
        out[f"first_ok_{variant}"] = ("\ufffd" not in first) and first.strip() != ""
    return out

token_stats("montagne", tok)

### Token statistics: what the columns mean

Each word is tokenized in two versions:
- **plain**: the word as is (`montagne`). This is what follows the opening quote in Wendler-style prompts.
- **space**: with a leading space (` montagne`). This is how the word appears mid-sentence.

Per-word columns (in `stats`):

| Column | Meaning |
|---|---|
| `n_tok_*` | number of tokens the word is split into |
| `first_id_*` | id of the first token, the one we measure in the logit lens |
| `first_str_*` | that first token as text |
| `first_ok_*` | True if the first token is usable: not only a space and not a broken byte piece (`�`) |
| `single` | True if the word is one token in either version |

Per-language summary (in `summary`):

| Column | Meaning |
|---|---|
| `n` | number of words |
| `single` | how many words are a single token |
| `mean_tok_plain` / `mean_tok_space` | average tokens per word; higher means the tokenizer chops the language more |
| `first_ok_plain` / `first_ok_space` | fraction of words with a usable first token (1.0 = all) |

Takeaway (Qwen3): Chinese is fine plain but not with a space (`first_ok_space` ≈ 0.04), so Wendler prompts use the plain version.


In [ ]:
stats = concepts.join(concepts["word"].apply(lambda w: pd.Series(token_stats(w, tok))))
stats["single"] = (stats["n_tok_plain"] == 1) | (stats["n_tok_space"] == 1)

summary = stats.groupby("lang").agg(
    n=("word", "size"),
    single=("single", "sum"),
    mean_tok_plain=("n_tok_plain", "mean"),
    mean_tok_space=("n_tok_space", "mean"),
    first_ok_plain=("first_ok_plain", "mean"),
    first_ok_space=("first_ok_space", "mean"),
).round(2)
summary

In [ ]:
stats.loc[~stats["first_ok_space"], ["lang", "word", "first_str_space"]]

In [ ]:
ambiguous = stats.duplicated(["lang", "word"], keep=False)
clean = stats[~ambiguous].copy()
print(len(stats), "->", len(clean)) 

In [ ]:
def select_concepts(stats, langs, variant="plain"):
    """Concepts present in every lang in `langs`, with usable and pairwise-distinct first tokens."""
    id_col, ok_col = f"first_id_{variant}", f"first_ok_{variant}"

    wide = stats.pivot(index="concept_id", columns="lang", values=["word", id_col, ok_col])
    wide = wide.dropna(subset=[("word", l) for l in langs])     
    n_all = len(wide)

    keep = pd.Series(True, index=wide.index)
    for l in langs:                                 
        keep &= wide[(ok_col, l)].astype(bool)
    for i, a in enumerate(langs):                 
        for b in langs[i + 1:]:
            keep &= wide[(id_col, a)] != wide[(id_col, b)]

    out = pd.DataFrame(index=wide.index[keep])
    for l in langs:
        out[l] = wide.loc[keep, ("word", l)]
        out[f"{l}_id"] = wide.loc[keep, (id_col, l)].astype(int)
    print(f"{' / '.join(langs)}: {n_all} concepts in all languages → {len(out)} kept")
    return out.reset_index(), wide.index[~keep]

fr_zh, dropped = select_concepts(clean, ["fra_Latn", "cmn_Hans", "eng_Latn"])
print("dropped:", list(dropped))
fr_zh.head()

In [ ]:
select_concepts(clean, ["deu_Latn", "cmn_Hans", "eng_Latn"]);  
select_concepts(clean, ["rus_Cyrl", "cmn_Hans", "eng_Latn"]);   
select_concepts(clean, ["deu_Latn", "fra_Latn", "eng_Latn"]);  

In [ ]:
from collections import defaultdict

def bytes_to_unicode():
    """GPT-2 / Qwen map: each of the 256 bytes is shown as one printable character (e.g. space → 'Ġ')."""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))

byte_decoder = {c: b for b, c in bytes_to_unicode().items()}     # reverse map: printable char → byte

def token_bytes(tok):
    """{token_id: the raw bytes that token stands for}."""
    out = {}
    for tid in range(len(tok)):
        piece = tok.convert_ids_to_tokens(tid)                   # e.g. 'Ġcave', or 'äºĳ' for 云
        try:
            out[tid] = bytes(byte_decoder[c] for c in piece)
        except KeyError:                                         # skip any token not in the byte alphabet
            pass
    return out

tb = token_bytes(tok)
bytes_to_ids = defaultdict(list)            # reverse lookup: bytes → list of ids that have them
for tid, b in tb.items():
    bytes_to_ids[b].append(tid)

print(len(tok), len(tb))                    
print(tb[25385], tb[99718])                 # b' cave'  b'\xe4\xba\x91' (云 = 3 bytes)

In [ ]:
def is_whole_chars(b):
    """True if the bytes form complete characters (no half-character at the end)."""
    try:
        b.decode("utf-8")
        return True
    except UnicodeDecodeError:
        return False

def prefix_ids(word, bytes_to_ids, allow_partial=False):
    """Every token id whose bytes are a prefix of `word` or of ' ' + `word`."""
    ids = set()
    for text in (word, " " + word):
        b = text.encode("utf-8")
        for k in range(1, len(b) + 1):
            p = b[:k]
            if p.strip() == b"":                              # just a space: says nothing about the word
                continue
            if not allow_partial and not is_whole_chars(p):   # half a character, e.g. b'\xe0'
                continue
            ids.update(bytes_to_ids.get(p, []))
    return sorted(ids)

def show(ids):
    return [tok.decode([i]) for i in ids]

In [ ]:
for w in ["cave", "云", "montagne", "पानी", "облако"]:
    print(f"{w:10s}", show(prefix_ids(w, bytes_to_ids)),
          "| with partial bytes:", len(prefix_ids(w, bytes_to_ids, allow_partial=True)))

In [ ]:
clean["prefix_ids"] = clean["word"].apply(lambda w: prefix_ids(w, bytes_to_ids))
clean["n_prefix"] = clean["prefix_ids"].apply(len)
display(clean.groupby("lang")["n_prefix"].agg(["mean", "min", "max"]).round(1))

# Wendler's rule: the two sets must not share any id
sets = clean.pivot(index="concept_id", columns="lang", values="prefix_ids")
for a, b in [("cmn_Hans", "eng_Latn"), ("fra_Latn", "eng_Latn"), ("deu_Latn", "eng_Latn"), ("rus_Cyrl", "eng_Latn")]:
    s = sets[[a, b]].dropna()
    overlap = s.apply(lambda r: len(set(r[a]) & set(r[b])) > 0, axis=1)
    print(f"{a} vs {b}: {overlap.sum()} / {len(s)} concepts share a prefix token")

In [ ]:
from latent_switch.concepts import load_wendler, drop_ambiguous, add_token_info, translation_set

concepts = drop_ambiguous(load_wendler(WENDLER))
stats = add_token_info(concepts, tok)                       # one tokenizer → token columns for every row
fr_zh = translation_set(stats, src="fra_Latn", tgt="cmn_Hans", latent="eng_Latn")
fr_zh.head()